In [18]:
import pandas as pd
import numpy as np

data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
}

df = pd.DataFrame(data)
print(df)

     Outlook Temperature Humidity    Wind PlayTennis
0      Sunny         Hot     High    Weak         No
1      Sunny         Hot     High  Strong         No
2   Overcast         Hot     High    Weak        Yes
3       Rain        Mild     High    Weak        Yes
4       Rain        Cool   Normal    Weak        Yes
5       Rain        Cool   Normal  Strong         No
6   Overcast        Cool   Normal  Strong        Yes
7      Sunny        Mild     High    Weak         No
8      Sunny        Cool   Normal    Weak        Yes
9       Rain        Mild   Normal    Weak        Yes
10     Sunny        Mild   Normal  Strong        Yes
11  Overcast        Mild     High  Strong        Yes
12  Overcast         Hot   Normal    Weak        Yes
13      Rain        Mild     High  Strong         No


In [ ]:
#defining the feature column and the target column
feature_cols = ['Outlook', 'Temperature', 'Humidity', 'Wind']
target_col = 'PlayTennis'

In [19]:
#entropy
def entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)
    total_count = len(target_col)

    ent = 0.0
    for count in counts:
        p = count / total_count
        if p > 0:
            ent -= p * np.log2(p)

    return ent

In [20]:
#info_gain
def info_gain(data, feature_col, target_col):
    total_entropy = entropy(data[target_col])
    values, counts = np.unique(data[feature_col], return_counts=True)
    weighted_entropy = 0.0
    for value, count in zip(values, counts):
        subset = data[data[feature_col] == value]
        subset_entropy = entropy(subset[target_col])
        weighted_entropy += (count / len(data)) * subset_entropy

    info_gain = total_entropy - weighted_entropy
    return info_gain

In [21]:
#id3 descision tree training
def id3(data, original_data, features, target_attribute_name, parent_node_class=None):

  # If all target_attribute_name values are the same, return that value
  if len(np.unique(data[target_attribute_name])) == 1:
    return np.unique(data[target_attribute_name])[0]

  # If dataset is empty, return the most common target attribute value in original_data
  if len(data) == 0:
    return np.unique(original_data[target_attribute_name])[np.argmax(
        np.unique(original_data[target_attribute_name], return_counts=True)[1])]

  # If no features left, return the parent node class
  if len(features) == 0:
    return parent_node_class

  # Determine the parent node class (most common target attribute value in current data)
  parent_node_class = np.unique(data[target_attribute_name])[np.argmax(
      np.unique(data[target_attribute_name], return_counts=True)[1])]

  # Calculate information gain for each feature
  gains = [info_gain(data, feature, target_attribute_name) for feature in features]
  best_feature_idx = np.argmax(gains)
  best_feature = features[best_feature_idx]

  tree = {best_feature: {}}

  # Remove the best feature from the list of features
  remaining_features = [f for f in features if f != best_feature]

  # Build subtree for each unique value of the best feature
  for value in np.unique(data[best_feature]):
    subset = data[data[best_feature] == value]
    subtree = id3(subset, original_data, remaining_features, target_attribute_name, parent_node_class)
    tree[best_feature][value] = subtree

  return tree

In [22]:
def print_tree(tree, indent=0):
    for key, value in tree.items():
        if isinstance(value, dict):
            print('  ' * indent + str(key) + ':')
            print_tree(value, indent + 1)
        else:
            print('  ' * indent + str(key) + ': ' + str(value))

In [23]:
print_tree(tree)

Outlook:
  Overcast: Yes
  Rain:
    Wind:
      Strong: No
      Weak: Yes
  Sunny:
    Humidity:
      High: No
      Normal: Yes
